# LeNet (1998) — 논문이 문장으로만 적은 자리를 재 본다LeCun · Bottou · Bengio · Haffner 의 **LeNet-5** 를 노트 5절의 표 그대로 PyTorch 로 세우고, 노트 14절이 「논문에 없다」고 적은 자리 다섯을 잰다.| 실험 | 묻는 것 | 노트의 어느 자리에 닿는가 ||---|---|---|| A | 표대로 세우면 학습되는 값이 정확히 60,000 이 되는가. 시험 오차가 몇인가 | 5절의 층 표와 15절 하나(60,000 과 340,908) || **B** | **픽셀을 고정 순열로 섞으면 합성곱 망과 완전연결 망이 각각 얼마나 무너지는가** | **14절 셋 — 논문이 SVM 을 두고 문장으로만 적고 재지 않았다** || **C** | **얼마나 옮겨도 견디는가** | **14절 넷 — 논문의 불변 범위 셋이 「추정된다」로만 적혀 있다** || D | 같은 연결 구조에서 가중치 공유만 끄면 파라미터와 오차가 얼마가 되는가 | 14절 하나 — 세 생각의 몫이 갈려 있지 않다 || E | 표 I 대신 C3 를 전부 연결하면 무엇이 달라지는가 | 14절 다섯 — 표 I 의 근거 둘에 숫자가 없다 |---## 돌리기 전에 밝혀 두는 것 넷**하나. 규모를 줄였다.** 논문은 학습 60,000 장을 20 바퀴 돌았고, 여기서는 **12,000 장 · 12 바퀴**다. 줄인 이유는 실험 다섯을 한 시간 안에 끝내려는 것이고, **줄인 채로도 B · D · E 의 비교는 성립한다** — 조건들이 같은 데이터 · 같은 크기 · 같은 최적화를 쓰기 때문이다. 다만 **절대 오차율을 논문의 0.95% 와 같은 칸에 놓지 않는다.****둘. 최적화가 논문의 것이 아니다.** 논문은 확률적 대각 레벤버그-마콰르트(부록 C)와 바퀴마다 내려가는 전역 학습률을 쓴다. 여기서는 **관성 붙은 확률적 기울기 하강**을 쓴다. 바꾼 이유는 A~E 가 묻는 것이 전부 **조건 사이의 차이**이고, 그 차이는 최적화를 고정해야 읽히기 때문이다. **그래서 A 의 절대 오차는 「논문의 절차를 재현한 값」이 아니다.****셋. 왜곡 학습을 하지 않는다.** 논문의 0.8% 는 왜곡 540,000 장을 더한 값이다. **C 가 묻는 것이 「왜곡을 안 넣고 배운 망이 얼마나 견디는가」라서** 일부러 넣지 않는다.**넷. 축소 규칙을 미리 적는다.** 실험 하나가 6 분을 넘으면 `FAST = True` 로 **학습 장수만** 절반으로 줄인다. 에폭 수와 망 크기는 줄이지 않는다 — 줄이면 「구조가 값을 못 했다」와 「둘 다 덜 배웠다」가 섞인다.**예비 실행을 한 번 했다는 것을 밝혀 둔다.** 학습률을 고르느라 seed 0 으로 합성곱 망을 한 번 미리 돌렸다. 0.02 에서는 첫 바퀴에 주저앉았고(오차 90.2% 에서 안 움직였다) 0.003 에서 12 바퀴에 1.88% 가 나왔다. **그래서 H1 의 범위(1~3%)는 그 실행을 보고 적은 것이고, H2~H5 는 그 실행에서 보지 않은 것이다.****판정 규칙.** seed 셋의 **변동 폭(최소~최대)을 먼저 적고, 그 폭보다 작은 차이로 순서를 매기지 않는다.** B 는 모델을 **자기 자신과** 견준다. C 의 「견디는 폭」은 **오차가 이동 0 일 때의 2 배를 넘지 않는 가장 큰 이동 픽셀 수**로 미리 정의한다.

In [ ]:
# ── 준비 ────────────────────────────────────────────────────────────────
import os, sys, time, math, platform
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

NOTEBOOK = "lenet_1998.ipynb"
RUN_ID   = time.strftime("%Y%m%d_%H%M%S")
HERE     = os.getcwd()
RESULTS  = os.path.join(HERE, "results")
FIGURES  = os.path.join(HERE, "figures")
DATA     = os.path.join(HERE, "data")
for d in (RESULTS, FIGURES, DATA):
    os.makedirs(d, exist_ok=True)

DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ENV = {
    "run_id": RUN_ID, "notebook": NOTEBOOK,
    "python": sys.version.split()[0], "torch": torch.__version__,
    "numpy": np.__version__, "pandas": pd.__version__,
    "device": (torch.cuda.get_device_name(0) if DEV.type == "cuda" else "cpu"),
    "platform": platform.platform(), "cwd": HERE,
}
for k, v in ENV.items():
    print("%-9s %s" % (k, v))


def save(df, name):
    """결과 CSV 에 재현 정보를 열로 붙여 저장한다."""
    df = df.copy()
    for k in ("run_id", "notebook", "python", "torch", "device"):
        df[k] = ENV[k]
    path = os.path.join(RESULTS, name)
    df.to_csv(path, index=False, encoding="utf-8-sig")
    print("저장:", name, df.shape)
    return df


def span(v):
    """변동 폭을 '최소~최대' 로 적는다. 판정 규칙이 요구하는 것."""
    v = np.asarray(v, float)
    return "%.3f~%.3f" % (v.min(), v.max())


# ── 이 실행의 손잡이 ─────────────────────────────────────────────────────
N_TRAIN  = 12_000     # 논문은 60,000. 줄인 이유는 맨 위에 적었다
N_TEST   = 10_000     # 논문도 시험은 10,000 장을 쓴다
EPOCHS   = 12         # 논문은 20 바퀴
BATCH    = 32
# 학습률은 모델마다 다르게 둔다. 두 모델의 벌점 크기가 다르기 때문이다 —
# RBF 출력은 84 차원 제곱 거리라 값이 수십에서 수백이고, 완전연결 대조군은 로짓이라 한 자릿수다.
# 같은 값을 쓰면 한쪽이 안 배워진다. 아래 값은 돌리기 전에 짧은 예비 실행으로 골랐고,
# **한 모델의 두 조건(원본 · 픽셀 섞음)에는 같은 학습률을 쓴다** — B 가 묻는 것이 모델 안에서의 변화라서다.
LR_CNN   = 0.003      # 0.02 에서는 첫 바퀴에 주저앉았다
LR_MLP   = 0.03
MOMENTUM = 0.9
SEEDS    = [0, 1, 2]
J_CONST  = 1.0        # 식 9 의 상수 j. 논문에 값이 없어 여기서 정한다
FAST     = False      # True 면 학습 장수만 절반으로 줄인다(축소 규칙)
if FAST:
    N_TRAIN = N_TRAIN // 2

print()
print("학습 %d 장 · 시험 %d 장 · %d 바퀴 · 묶음 %d · seed %s"
      % (N_TRAIN, N_TEST, EPOCHS, BATCH, SEEDS))

## 0. 데이터 — MNIST 를 논문의 입력 모양으로 맞춘다논문 II-B 가 적는 대로 **28×28 을 32×32 로 넓히고**, **배경을 −0.1 · 글자를 1.175** 로 둔다. 그러면 입력의 평균이 0 근처, 분산이 1 근처가 된다.받지 못하면 **그 사실을 적고 멈춘다** — 합성 데이터로 바꿔치기하면 결과를 논문 이야기와 잇지 못한다.

In [ ]:
# ── 데이터 ──────────────────────────────────────────────────────────────
from torchvision import datasets

BG, FG = -0.1, 1.175     # 논문 II-B

def load(train, n, seed=12345):
    ds = datasets.MNIST(DATA, train=train, download=True)
    X = ds.data.float() / 255.0                     # (N, 28, 28), 배경 0 · 글자 1
    y = ds.targets.clone()
    g = torch.Generator().manual_seed(seed)
    idx = torch.randperm(X.shape[0], generator=g)[:n]
    X, y = X[idx], y[idx]
    X = F.pad(X.unsqueeze(1), (2, 2, 2, 2), value=0.0)   # 32 x 32 로 넓힌다
    X = X * (FG - BG) + BG                               # 배경 -0.1 · 글자 1.175
    return X.to(DEV), y.to(DEV)


Xtr, ytr = load(True,  N_TRAIN)
Xte, yte = load(False, N_TEST, seed=777)

print("학습", tuple(Xtr.shape), "시험", tuple(Xte.shape))
print("픽셀 범위 %.3f ~ %.3f · 평균 %.3f · 표준편차 %.3f"
      % (Xtr.min(), Xtr.max(), Xtr.mean(), Xtr.std()))
print("부류별 장수", torch.bincount(ytr, minlength=10).tolist())

## 1. 구현 — 노트 5절의 표를 그대로 세운다- **누름 함수** $f(a) = 1.7159\tanh(\tfrac{2}{3}a)$ 와 초기화 $[-2.4/F_i,\ 2.4/F_i]$ 는 부록 A 그대로다.- **부분 표본 뽑기**는 2×2 를 **더하고** 학습되는 계수를 곱하고 바이어스를 더한 뒤 누름 함수를 지난다. 평균 풀링에 4 를 곱해 합을 만든다.- **C3** 는 표 I 의 연결만 살리는 마스크를 가중치에 곱해 만든다. `full=True` 면 전부 연결이 되고, 그것이 실험 E 다.- **출력**은 유클리드 RBF 다. 파라미터 벡터는 7×12 비트맵의 ±1 이고 **배우지 않는다.** 값이 작을수록 그 부류다.- **손실**은 식 9 다. `logadd` 항에 쓰레기 부류 상수 $j$ 가 들어간다.- **완전연결 대조군도 같은 손실을 쓴다** — 출력을 벌점으로 읽으면(로짓에 음수를 붙이면) 식 9 가 그대로 적용된다.- **학습률만 모델마다 다르다.** 두 모델의 벌점 크기가 다르기 때문이다(RBF 는 84 차원 제곱 거리, 로짓은 한 자릿수). **한 모델의 두 조건에는 같은 학습률을 쓴다** — B 가 묻는 것이 모델 안에서의 변화라서다.

In [ ]:
# ── 구현 ────────────────────────────────────────────────────────────────
A_SQ, S_SQ = 1.7159, 2.0 / 3.0

def squash(x):
    return A_SQ * torch.tanh(S_SQ * x)


def init_uniform_(t, fan_in):
    b = 2.4 / float(fan_in)
    nn.init.uniform_(t, -b, b)


# 표 I — C3 의 특징 지도 열여섯이 S2 의 어느 장을 받는가
TABLE1 = [
    [0, 1, 2], [1, 2, 3], [2, 3, 4], [3, 4, 5], [0, 4, 5], [0, 1, 5],
    [0, 1, 2, 3], [1, 2, 3, 4], [2, 3, 4, 5], [0, 3, 4, 5], [0, 1, 4, 5], [0, 1, 2, 5],
    [0, 1, 3, 4], [1, 2, 4, 5], [0, 2, 3, 5],
    [0, 1, 2, 3, 4, 5],
]

def table1_mask(full=False):
    m = torch.zeros(16, 6, 1, 1)
    for c, rows in enumerate(TABLE1):
        if full:
            m[c, :, 0, 0] = 1.0
        else:
            for r in rows:
                m[c, r, 0, 0] = 1.0
    return m


class Subsample(nn.Module):
    """논문의 부분 표본 뽑기. 최댓값 풀링이 아니다 — 더하고 계수를 곱하고 바이어스를 더한다."""
    def __init__(self, ch):
        super().__init__()
        self.coef = nn.Parameter(torch.ones(ch))
        self.bias = nn.Parameter(torch.zeros(ch))

    def forward(self, x):
        s = F.avg_pool2d(x, 2) * 4.0                      # 평균 x 4 = 넷의 합
        return squash(s * self.coef.view(1, -1, 1, 1) + self.bias.view(1, -1, 1, 1))

    def n_learn(self):
        return self.coef.numel() + self.bias.numel()


class MaskedConv(nn.Module):
    """가중치를 공유하는 합성곱. mask 로 어느 입력 장을 받을지 고른다."""
    def __init__(self, mask, k=5):
        super().__init__()
        o, i = mask.shape[0], mask.shape[1]
        self.k = k
        self.register_buffer("mask", mask)
        self.weight = nn.Parameter(torch.empty(o, i, k, k))
        self.bias = nn.Parameter(torch.zeros(o))
        fan = int(mask.sum(dim=1).max().item()) * k * k   # 팬인이 장마다 다르면 가장 큰 것으로 잡는다
        init_uniform_(self.weight, fan)

    def forward(self, x):
        return F.conv2d(x, self.weight * self.mask, self.bias)

    def n_learn(self):
        return int(self.mask.sum().item()) * self.k * self.k + self.bias.numel()


class LocalConv(nn.Module):
    """가중치 공유를 끈 국소 연결층 — 자리마다 가중치를 따로 둔다. 실험 D 가 쓴다."""
    def __init__(self, in_ch, out_ch, in_hw, k=5, mask=None):
        super().__init__()
        self.k = k
        self.out_hw = in_hw - k + 1
        L = self.out_hw ** 2
        self.weight = nn.Parameter(torch.empty(out_ch, L, in_ch * k * k))
        self.bias = nn.Parameter(torch.zeros(out_ch, L))
        init_uniform_(self.weight, in_ch * k * k)
        if mask is None:
            mask = torch.ones(out_ch, in_ch, 1, 1)
        m = mask.view(out_ch, in_ch, 1, 1).repeat(1, 1, k, k).reshape(out_ch, 1, in_ch * k * k)
        self.register_buffer("mask", m)

    def forward(self, x):
        p = F.unfold(x, self.k)                                   # (N, in_ch*k*k, L)
        out = torch.einsum("nkl,olk->nol", p, self.weight * self.mask)
        out = out + self.bias.unsqueeze(0)
        return out.view(x.shape[0], -1, self.out_hw, self.out_hw)

    def n_learn(self):
        return int(self.mask.sum().item()) * self.weight.shape[1] + self.bias.numel()


# ── 출력 RBF — 7 x 12 비트맵을 ±1 로 펴서 고정한다 ──────────────────────
DIGIT_BITMAP = {
    0: ["0111100", "1100110", "1100110", "1100110", "1100110", "1100110",
        "1100110", "1100110", "1100110", "1100110", "0111100", "0000000"],
    1: ["0011000", "0111000", "1111000", "0011000", "0011000", "0011000",
        "0011000", "0011000", "0011000", "0011000", "1111110", "0000000"],
    2: ["0111100", "1100110", "0000110", "0000110", "0001100", "0011000",
        "0110000", "1100000", "1100000", "1100000", "1111110", "0000000"],
    3: ["0111100", "1100110", "0000110", "0000110", "0001100", "0011000",
        "0001100", "0000110", "0000110", "1100110", "0111100", "0000000"],
    4: ["0001100", "0011100", "0111100", "0110100", "1100100", "1100100",
        "1111110", "0000100", "0000100", "0000100", "0000100", "0000000"],
    5: ["1111110", "1100000", "1100000", "1111100", "1100110", "0000110",
        "0000110", "0000110", "0000110", "1100110", "0111100", "0000000"],
    6: ["0011100", "0110000", "1100000", "1100000", "1111100", "1100110",
        "1100110", "1100110", "1100110", "1100110", "0111100", "0000000"],
    7: ["1111110", "0000110", "0000110", "0001100", "0001100", "0011000",
        "0011000", "0110000", "0110000", "0110000", "0110000", "0000000"],
    8: ["0111100", "1100110", "1100110", "1100110", "0111100", "0111100",
        "1100110", "1100110", "1100110", "1100110", "0111100", "0000000"],
    9: ["0111100", "1100110", "1100110", "1100110", "1100110", "0111110",
        "0000110", "0000110", "0000110", "0011000", "0111000", "0000000"],
}

def make_protos():
    rows = []
    for d in range(10):
        v = [1.0 if ch == "1" else -1.0 for line in DIGIT_BITMAP[d] for ch in line]
        assert len(v) == 84, len(v)
        rows.append(v)
    return torch.tensor(rows, dtype=torch.float32)


class RBFOutput(nn.Module):
    """식 7. 출력이 작을수록 그 부류다. 파라미터는 고정이라 학습되는 값에 들어가지 않는다."""
    def __init__(self):
        super().__init__()
        self.register_buffer("proto", make_protos())

    def forward(self, x):
        return ((x.unsqueeze(1) - self.proto.unsqueeze(0)) ** 2).sum(-1)

    def n_learn(self):
        return 0


class LeNet5(nn.Module):
    def __init__(self, c3_full=False, untie=False):
        super().__init__()
        mask = table1_mask(full=c3_full)
        if untie:
            self.c1 = LocalConv(1, 6, 32)
            self.c3 = LocalConv(6, 16, 14, mask=mask)
        else:
            self.c1 = MaskedConv(torch.ones(6, 1, 1, 1))
            self.c3 = MaskedConv(mask)
        self.s2 = Subsample(6)
        self.s4 = Subsample(16)
        self.c5 = nn.Linear(16 * 5 * 5, 120)
        self.f6 = nn.Linear(120, 84)
        init_uniform_(self.c5.weight, 16 * 5 * 5)
        init_uniform_(self.f6.weight, 120)
        nn.init.zeros_(self.c5.bias)
        nn.init.zeros_(self.f6.bias)
        self.out = RBFOutput()

    def forward(self, x):
        x = squash(self.c1(x))
        x = self.s2(x)
        x = squash(self.c3(x))
        x = self.s4(x)
        x = squash(self.c5(x.flatten(1)))
        x = squash(self.f6(x))
        return self.out(x)                # 벌점 — 작을수록 그 부류

    def learn_table(self):
        return [
            ("C1", self.c1.n_learn()),
            ("S2", self.s2.n_learn()),
            ("C3", self.c3.n_learn()),
            ("S4", self.s4.n_learn()),
            ("C5", self.c5.weight.numel() + self.c5.bias.numel()),
            ("F6", self.f6.weight.numel() + self.f6.bias.numel()),
            ("출력 RBF", self.out.n_learn()),
        ]


class MLP(nn.Module):
    """완전연결 대조군 1024-300-10. 출력을 벌점으로 읽어 같은 손실을 쓴다."""
    def __init__(self, hidden=300):
        super().__init__()
        self.l1 = nn.Linear(32 * 32, hidden)
        self.l2 = nn.Linear(hidden, 10)
        init_uniform_(self.l1.weight, 32 * 32)
        init_uniform_(self.l2.weight, hidden)
        nn.init.zeros_(self.l1.bias)
        nn.init.zeros_(self.l2.bias)

    def forward(self, x):
        h = squash(self.l1(x.flatten(1)))
        return -self.l2(h)                # 부호를 뒤집어 벌점으로 읽는다

    def learn_table(self):
        return [("층 1", self.l1.weight.numel() + self.l1.bias.numel()),
                ("층 2", self.l2.weight.numel() + self.l2.bias.numel())]


def loss_eq9(pen, target, j=None):
    """식 9 — 정답 부류의 벌점 + log( e^-j + Σ e^-y_i )."""
    j = J_CONST if j is None else j
    yc = pen.gather(1, target.view(-1, 1)).squeeze(1)
    rubbish = torch.full((pen.shape[0], 1), -float(j), device=pen.device)
    lse = torch.logsumexp(torch.cat([-pen, rubbish], dim=1), dim=1)
    return (yc + lse).mean()


@torch.no_grad()
def error_pct(model, X, y, bs=1000):
    model.eval()
    wrong = 0
    for i in range(0, X.shape[0], bs):
        pred = model(X[i:i + bs]).argmin(1)       # 벌점이 가장 작은 부류
        wrong += (pred != y[i:i + bs]).sum().item()
    return 100.0 * wrong / X.shape[0]


def train_eval(factory, Xa, ya, Xb, yb, seed=0, epochs=None, tag="", lr=None):
    """seed 를 먼저 박고 망을 세운 뒤 배운다. 초기화까지 seed 에 걸리게 하려는 것이다."""
    epochs = EPOCHS if epochs is None else epochs
    lr = LR_CNN if lr is None else lr
    torch.manual_seed(seed)
    model = factory().to(DEV)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=MOMENTUM)
    sch = torch.optim.lr_scheduler.StepLR(opt, step_size=max(1, epochs // 3), gamma=0.3)
    n = Xa.shape[0]
    t0 = time.perf_counter()
    for ep in range(epochs):
        model.train()
        perm = torch.randperm(n, device=DEV)
        for i in range(0, n, BATCH):
            idx = perm[i:i + BATCH]
            loss = loss_eq9(model(Xa[idx]), ya[idx])
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
        sch.step()
    sec = time.perf_counter() - t0
    te, tr = error_pct(model, Xb, yb), error_pct(model, Xa, ya)
    print("  %-26s seed %d · 시험 %.2f%% · 학습 %.2f%% · %.0f 초" % (tag, seed, te, tr, sec))
    return model, te, tr, sec

### 구현이 맞는지 먼저 확인한다**둘을 본다.** (가) 층별 학습되는 값이 노트 5절의 표와 같은가. (나) 역전파 기울기가 유한 차분과 맞는가.**(가) 가 틀리면 그 아래 결과를 보지 않는다** — 다른 망을 재고 있다는 뜻이기 때문이다.

In [ ]:
# ── 검증 ────────────────────────────────────────────────────────────────
PAPER = [("C1", 156), ("S2", 12), ("C3", 1516), ("S4", 32), ("C5", 48120), ("F6", 10164),
         ("출력 RBF", 0)]

net = LeNet5().to(DEV)
tbl = net.learn_table()
rows = []
ok_all = True
for (name, got), (_, want) in zip(tbl, PAPER):
    ok = (got == want)
    ok_all = ok_all and ok
    rows.append({"층": name, "센 값": got, "논문": want, "같은가": ok})
dfV = pd.DataFrame(rows)
print(dfV.to_string(index=False))
total = sum(g for _, g in tbl)
print()
print("합계 %s · 논문 60,000 · %s" % (format(total, ","), "같다" if total == 60000 else "다르다"))
print("torch 가 세는 파라미터 수 %s (마스크로 0 이 된 자리를 포함한다)"
      % format(sum(p.numel() for p in net.parameters()), ","))
assert ok_all and total == 60000, "층별 값이 노트 5절의 표와 다르다 — 여기서 멈춘다"

# (나) 유한 차분
torch.manual_seed(0)
xb, yb_ = Xtr[:16], ytr[:16]
net.zero_grad()
loss_eq9(net(xb), yb_).backward()
p = net.c3.weight
i = (0, 0, 2, 2)
g_analytic = p.grad[i].item()
eps = 1e-3
with torch.no_grad():
    p[i] += eps
    lp = loss_eq9(net(xb), yb_).item()
    p[i] -= 2 * eps
    lm = loss_eq9(net(xb), yb_).item()
    p[i] += eps
g_numeric = (lp - lm) / (2 * eps)
print()
print("C3 가중치 하나 · 역전파 %.6f · 유한 차분 %.6f · 차이 %.2e"
      % (g_analytic, g_numeric, abs(g_analytic - g_numeric)))
save(dfV, "V_param_count.csv")

## A. 표대로 세운 LeNet-5 를 배운다**H1**: 학습되는 값이 정확히 60,000 이고, 시험 오차가 이 축소 규모에서 1% 에서 3% 사이에 든다.여기서 배운 망 셋(seed 0 · 1 · 2)을 **C 가 그대로 다시 쓴다.** C 는 다시 배우지 않고 시험 이미지만 옮긴다.

In [ ]:
# ── A. 기준 망 ──────────────────────────────────────────────────────────
print("[A] LeNet-5 — 표 I 연결 · 가중치 공유")
base_models, rowsA = [], []
for s in SEEDS:
    m, te, tr, sec = train_eval(lambda: LeNet5(), Xtr, ytr, Xte, yte, seed=s, tag="LeNet-5")
    base_models.append(m)
    rowsA.append({"model": "LeNet-5", "seed": s, "test_err": te, "train_err": tr,
                  "n_learn": sum(g for _, g in m.learn_table()), "seconds": sec})
dfA = pd.DataFrame(rowsA)
print()
print("시험 오차 중앙값 %.2f%% · 변동 폭 %s" % (dfA.test_err.median(), span(dfA.test_err)))
dfA = save(dfA, "A_baseline.csv")

## B. 픽셀을 고정 순열로 섞는다**노트 14절 셋이 가리키는 자리다.** 논문 III-D 는 SVM 을 두고 「픽셀을 고정된 대응으로 섞어도 똑같이 잘할 것」이라고 적지만 **재 보지 않았다.****학습과 시험에 같은 순열을 건다.** 곧 과제의 난이도는 그대로이고, 2차원 이웃 관계만 사라진다.**각 모델을 자기 자신과 견준다.** 두 모델의 절대값을 나란히 놓지 않는다 — 파라미터 수도 출력층도 학습률도 다르다.

In [ ]:
# ── B. 픽셀 순열 ────────────────────────────────────────────────────────
g = torch.Generator().manual_seed(4242)
PERM = torch.randperm(32 * 32, generator=g).to(DEV)

def permute_pixels(X):
    return X.flatten(1)[:, PERM].view(-1, 1, 32, 32)

Xtr_p, Xte_p = permute_pixels(Xtr), permute_pixels(Xte)
print("순열 예시(앞 8개):", PERM[:8].tolist())

rowsB = []
print()
print("[B] 섞기 전과 뒤")
for name, fac, lr in [("LeNet-5", lambda: LeNet5(), LR_CNN),
                      ("완전연결 1024-300-10", lambda: MLP(), LR_MLP)]:
    for cond, Xa, Xb in [("원본", Xtr, Xte), ("픽셀 섞음", Xtr_p, Xte_p)]:
        for s in SEEDS:
            _, te, tr, sec = train_eval(fac, Xa, ytr, Xb, yte, seed=s, lr=lr,
                                        tag="%s · %s" % (name, cond))
            rowsB.append({"model": name, "cond": cond, "seed": s, "lr": lr,
                          "test_err": te, "train_err": tr, "seconds": sec})
dfB = pd.DataFrame(rowsB)
print()
for name in dfB.model.unique():
    a = dfB[(dfB.model == name) & (dfB.cond == "원본")].test_err.values
    b = dfB[(dfB.model == name) & (dfB.cond == "픽셀 섞음")].test_err.values
    widest = max(a.max() - a.min(), b.max() - b.min())
    print("  %-22s 원본 %.2f%% (%s) · 섞음 %.2f%% (%s) · 차이 %+.2f · 변동 폭 %.2f -> %s"
          % (name, np.median(a), span(a), np.median(b), span(b),
             np.median(b) - np.median(a), widest,
             "폭보다 크다" if abs(np.median(b) - np.median(a)) > widest else "폭 안이다"))
dfB = save(dfB, "B_permutation.csv")

## C. 얼마나 옮겨도 견디는가**노트 14절 넷이 가리키는 자리다.** 논문은 「세로 이동 글자 높이의 절반쯤」이라고 적고 끝난다. MNIST 의 글자는 20 픽셀 칸에 맞춰져 있으므로 **그 절반은 10 픽셀**로 읽힌다.**A 에서 배운 망을 다시 쓰고 시험 이미지만 옮긴다.** 배경으로 채우고 잘라내므로 가장자리가 감기지 않는다.**견디는 폭을 미리 정의해 두었다** — 오차가 이동 0 일 때의 **2 배를 넘지 않는 가장 큰 이동 픽셀 수**다.

In [ ]:
# ── C. 이동 ─────────────────────────────────────────────────────────────
def shift_images(X, dx=0, dy=0):
    out = torch.full_like(X, BG)
    H = X.shape[-1]
    xs0, xs1 = max(0, dx), min(H, H + dx)
    ys0, ys1 = max(0, dy), min(H, H + dy)
    out[:, :, ys0:ys1, xs0:xs1] = X[:, :, ys0 - dy:ys1 - dy, xs0 - dx:xs1 - dx]
    return out


SHIFTS = list(range(-8, 9))
rowsC = []
print("[C] 시험 이미지를 옮긴다 (A 의 망을 다시 쓴다)")
for si, m in zip(SEEDS, base_models):
    for axis in ("가로", "세로"):
        for d in SHIFTS:
            Xs = shift_images(Xte, dx=(d if axis == "가로" else 0),
                              dy=(d if axis == "세로" else 0))
            rowsC.append({"seed": si, "axis": axis, "shift_px": d,
                          "test_err": error_pct(m, Xs, yte)})
dfC = pd.DataFrame(rowsC)

base_err = dfC[dfC.shift_px == 0].test_err.median()
print("  이동 0 일 때 오차 %.2f%% · 판정선(2 배) %.2f%%" % (base_err, 2 * base_err))
tol = {}
for axis in ("가로", "세로"):
    piv = dfC[dfC.axis == axis].groupby("shift_px").test_err.median()
    k = 0
    while k + 1 <= 8 and max(piv.get(k + 1, 1e9), piv.get(-(k + 1), 1e9)) <= 2 * base_err:
        k += 1
    tol[axis] = k
    print("  %s 로 견디는 폭 ±%d 픽셀" % (axis, k))
    print("   ", " ".join("%+d:%.1f" % (d, piv[d]) for d in SHIFTS))
dfC = save(dfC, "C_shift.csv")

In [ ]:
# ── C. 그림 ─────────────────────────────────────────────────────────────
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# 그림 안의 글자는 영어로 둔다 — Colab 에 한글 글꼴이 없어 네모로 깨지기 때문이다.
fig, ax = plt.subplots(figsize=(7.2, 4.0))
for axis, lab, style in [("가로", "horizontal", "-o"), ("세로", "vertical", "--s")]:
    piv = dfC[dfC.axis == axis].groupby("shift_px").test_err.median()
    ax.plot(piv.index, piv.values, style, ms=4, label="%s shift" % lab)
ax.axhline(2 * base_err, color="crimson", lw=1.0, ls=":", label="threshold = 2x baseline")
for v in (-10, 10):
    ax.axvline(v, color="gray", lw=1.0, ls=":")
ax.text(10.3, ax.get_ylim()[1] * 0.86, "paper's estimate\n(half the character height)",
        fontsize=8, color="gray")
ax.set_xlabel("shift (pixels)")
ax.set_ylabel("test error (%)")
ax.set_title("C. Shift tolerance (median over seeds %s)" % SEEDS)
ax.grid(alpha=.3)
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(os.path.join(FIGURES, "C_shift.png"), dpi=140)
plt.close(fig)
print("저장: figures/C_shift.png")

## D. 가중치 공유만 끈다**노트 14절 하나가 가리키는 자리다.** 연결 구조(국소 수용장 · 표 I · 부분 표본)는 그대로 두고 **공유만 끈다.** 곧 자리마다 가중치를 따로 둔다.학습되는 값이 60,000 에서 **332,232** 로 는다(C1 122,304 + S2 12 + C3 151,600 + S4 32 + C5 48,120 + F6 10,164). 이 수가 맞는지도 함께 확인한다.

In [ ]:
# ── D. 공유를 끈다 ──────────────────────────────────────────────────────
probe = LeNet5(untie=True).to(DEV)
print("공유를 끈 망의 층별 학습되는 값:", probe.learn_table())
n_untied = sum(g for _, g in probe.learn_table())
print("합계 %s (공유하면 %s · %.1f 배)"
      % (format(n_untied, ","), format(60000, ","), n_untied / 60000.0))
del probe

rowsD = []
print()
print("[D] 공유를 끈 망")
for s in SEEDS:
    m, te, tr, sec = train_eval(lambda: LeNet5(untie=True), Xtr, ytr, Xte, yte,
                                seed=s, tag="공유 끔")
    rowsD.append({"model": "공유 끔", "seed": s, "test_err": te, "train_err": tr,
                  "n_learn": n_untied, "seconds": sec})
dfD = pd.concat([dfA[["model", "seed", "test_err", "train_err", "n_learn", "seconds"]],
                 pd.DataFrame(rowsD)], ignore_index=True)
a = dfD[dfD.model == "LeNet-5"].test_err.values
b = dfD[dfD.model == "공유 끔"].test_err.values
widest = max(a.max() - a.min(), b.max() - b.min())
print()
print("  공유 %.2f%% (%s) · 공유 끔 %.2f%% (%s) · 차이 %+.2f · 변동 폭 %.2f -> %s"
      % (np.median(a), span(a), np.median(b), span(b), np.median(b) - np.median(a), widest,
         "폭보다 크다" if abs(np.median(b) - np.median(a)) > widest else "폭 안이다"))
dfD = save(dfD, "D_weight_sharing.csv")

## E. 표 I 대신 C3 를 전부 연결한다**노트 14절 다섯이 가리키는 자리다.** 논문은 비완전 연결의 근거로 (가) 연결 수를 다스린다 (나) 대칭을 깬다 를 드는데 **전부 연결과 견준 숫자가 없다.**학습되는 값이 C3 에서 1,516 에서 2,416 으로 늘어 망 전체가 60,000 에서 60,900 이 된다.

In [ ]:
# ── E. 표 I 대 전부 연결 ────────────────────────────────────────────────
probe = LeNet5(c3_full=True).to(DEV)
n_full = sum(g for _, g in probe.learn_table())
print("전부 연결의 층별 학습되는 값:", probe.learn_table())
print("합계 %s (표 I 은 %s · 차이 %s)"
      % (format(n_full, ","), format(60000, ","), format(n_full - 60000, ",")))
del probe

rowsE = []
print()
print("[E] C3 를 전부 연결")
for s in SEEDS:
    m, te, tr, sec = train_eval(lambda: LeNet5(c3_full=True), Xtr, ytr, Xte, yte,
                                seed=s, tag="C3 전부 연결")
    rowsE.append({"model": "C3 전부 연결", "seed": s, "test_err": te, "train_err": tr,
                  "n_learn": n_full, "seconds": sec})
dfE = pd.concat([dfA[["model", "seed", "test_err", "train_err", "n_learn", "seconds"]],
                 pd.DataFrame(rowsE)], ignore_index=True)
a = dfE[dfE.model == "LeNet-5"].test_err.values
b = dfE[dfE.model == "C3 전부 연결"].test_err.values
widest_E = max(a.max() - a.min(), b.max() - b.min())
print()
print("  표 I %.2f%% (%s) · 전부 연결 %.2f%% (%s) · 차이 %+.2f · 변동 폭 %.2f -> %s"
      % (np.median(a), span(a), np.median(b), span(b), np.median(b) - np.median(a), widest_E,
         "폭보다 크다" if abs(np.median(b) - np.median(a)) > widest_E else "폭 안이다"))
dfE = save(dfE, "E_c3_connection.csv")

In [ ]:
# ── B · D · E 그림 ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(11.0, 4.0))

EN = {"LeNet-5": "LeNet-5", "완전연결 1024-300-10": "fully connected 1024-300-10",
      "공유 끔": "weights untied", "C3 전부 연결": "C3 fully connected"}

ax = axes[0]
labels, before, after = [], [], []
for name in dfB.model.unique():
    labels.append(EN[name])
    before.append(dfB[(dfB.model == name) & (dfB.cond == "원본")].test_err.median())
    after.append(dfB[(dfB.model == name) & (dfB.cond == "픽셀 섞음")].test_err.median())
xs = np.arange(len(labels))
ax.bar(xs - 0.18, before, 0.36, label="original pixels")
ax.bar(xs + 0.18, after, 0.36, label="fixed permutation")
ax.set_xticks(xs)
ax.set_xticklabels(labels, fontsize=8)
ax.set_ylabel("test error (%)")
ax.set_title("B. Each model compared with itself")
ax.grid(alpha=.3, axis="y")
ax.legend(fontsize=8)

ax = axes[1]
names = ["LeNet-5", "공유 끔", "C3 전부 연결"]
vals, errs = [], []
src = pd.concat([dfD, dfE[dfE.model == "C3 전부 연결"]], ignore_index=True)
for n in names:
    v = src[src.model == n].test_err.values
    vals.append(np.median(v))
    errs.append([np.median(v) - v.min(), v.max() - np.median(v)])
ax.bar([EN[n] for n in names], vals, 0.5, yerr=np.array(errs).T, capsize=4)
ax.set_ylabel("test error (%)")
ax.set_title("D, E. Bars show the min-max span over seeds %s" % SEEDS)
ax.grid(alpha=.3, axis="y")
ax.tick_params(axis="x", labelsize=8)
for i, n in enumerate(names):
    k = src[src.model == n].n_learn.iloc[0]
    ax.text(i, vals[i], " %s params" % format(int(k), ","), ha="center", va="bottom", fontsize=8)

fig.tight_layout()
fig.savefig(os.path.join(FIGURES, "BDE_summary.png"), dpi=140)
plt.close(fig)
print("저장: figures/BDE_summary.png")

## 요약 — 아래 출력을 그대로 `README.md` 의 결과 칸에 옮긴다**가설마다 맞았는지 틀렸는지를 적는다.** 틀렸으면 틀렸다고 적는다. **변동 폭보다 작은 차이로는 순서를 매기지 않는다.**

In [ ]:
# ── 요약 ────────────────────────────────────────────────────────────────
print("=" * 88)
print("LeNet (1998) 실험 요약 · run_id %s" % RUN_ID)
print("설정: 학습 %d · 시험 %d · %d 바퀴 · 묶음 %d · seed %s · 장치 %s"
      % (N_TRAIN, N_TEST, EPOCHS, BATCH, SEEDS, ENV["device"]))
print("학습률: 합성곱 %.3f · 완전연결 %.3f (한 모델의 두 조건에는 같은 값을 썼다)" % (LR_CNN, LR_MLP))
print("주의: 최적화가 논문의 것이 아니고 왜곡 학습을 하지 않았다. 절대 오차를 논문의 0.95% 와 같은 칸에 두지 않는다.")
print("=" * 88)

print()
print("[A] 표대로 세운 LeNet-5 — H1")
print("  학습되는 값 합계 %s · 논문 60,000 · %s"
      % (format(int(dfA.n_learn.iloc[0]), ","), "같다" if dfA.n_learn.iloc[0] == 60000 else "다르다"))
_a = dfA.test_err.values
print("  시험 오차 %.2f%% (%s) · 학습 오차 %.2f%%" % (np.median(_a), span(_a), dfA.train_err.median()))
_h1 = (dfA.n_learn.iloc[0] == 60000) and (1.0 <= np.median(_a) <= 3.0)
print("  H1 판정:", "맞았다" if _h1 else
      ("절반만 맞았다 — 파라미터는 맞는데 오차가 예상 범위(1~3%) 밖이다"
       if dfA.n_learn.iloc[0] == 60000 else "틀렸다 — 파라미터 수부터 다르다"))

print()
print("[B] 픽셀 순열 — H2  (논문이 문장으로만 적은 자리)")
_gaps = {}
for name in dfB.model.unique():
    a = dfB[(dfB.model == name) & (dfB.cond == "원본")].test_err.values
    b = dfB[(dfB.model == name) & (dfB.cond == "픽셀 섞음")].test_err.values
    w = max(a.max() - a.min(), b.max() - b.min())
    _gaps[name] = (np.median(b) - np.median(a), w)
    print("  %-22s %.2f%% -> %.2f%% · 차이 %+.2f · 변동 폭 %.2f -> %s"
          % (name, np.median(a), np.median(b), np.median(b) - np.median(a), w,
             "폭보다 크다" if abs(np.median(b) - np.median(a)) > w else "폭 안이다"))
_cnn = _gaps["LeNet-5"]
_mlp = _gaps["완전연결 1024-300-10"]
if _cnn[0] > _cnn[1] and abs(_mlp[0]) <= _mlp[1]:
    print("  H2 판정: 맞았다 — 합성곱 쪽만 변동 폭보다 크게 올랐다.")
elif _cnn[0] > _cnn[1] and _mlp[0] > _mlp[1]:
    print("  H2 판정: 절반만 맞았다 — 둘 다 올랐다. 그러면 순열이 과제 자체를 어렵게 한 몫이 섞여 있다.")
else:
    print("  H2 판정: 틀렸다 — 합성곱 쪽이 변동 폭보다 크게 오르지 않았다.")
    print("           이 규모에서는 2차원 이웃 관계가 오차에 값을 덜 한 것이다.")

print()
print("[C] 이동 — H3  (논문의 「추정된다」에 숫자를 붙인다)")
print("  기준 오차 %.2f%% · 판정선 %.2f%% (정의: 기준의 2 배)" % (base_err, 2 * base_err))
for axis in ("가로", "세로"):
    print("  %s 로 견디는 폭 ±%d 픽셀" % (axis, tol[axis]))
print("  논문이 적은 세로 이동 범위는 글자 높이(20 픽셀)의 절반, 곧 ±10 픽셀로 읽힌다.")
print("  H3 판정:", "맞았다 — ±10 픽셀까지 판정선 안이다" if tol["세로"] >= 10 else
      "틀렸다 — 견디는 폭이 ±%d 픽셀로 논문의 추정보다 좁다. 이 망은 왜곡 학습을 하지 않았다는 것을 함께 적는다" % tol["세로"])

print()
print("[D] 가중치 공유의 몫 — H4")
a = dfD[dfD.model == "LeNet-5"].test_err.values
b = dfD[dfD.model == "공유 끔"].test_err.values
w = max(a.max() - a.min(), b.max() - b.min())
print("  공유 %.2f%% (%s · 학습되는 값 %s) · 공유 끔 %.2f%% (%s · %s)"
      % (np.median(a), span(a), format(60000, ","), np.median(b), span(b), format(int(n_untied), ",")))
print("  차이 %+.2f · 변동 폭 %.2f" % (np.median(b) - np.median(a), w))
if np.median(b) - np.median(a) > w:
    print("  H4 판정: 맞았다 — 공유를 끄면 파라미터가 %.1f 배가 되고 오차도 올라간다." % (n_untied / 60000.0))
elif abs(np.median(b) - np.median(a)) <= w:
    print("  H4 판정: 절반만 맞았다 — 파라미터는 늘었는데 오차 차이가 변동 폭 안이다.")
    print("           그러면 이 규모에서 공유의 값은 오차가 아니라 파라미터 수 쪽에 있다.")
else:
    print("  H4 판정: 틀렸다 — 공유를 끈 쪽이 오히려 낫고 그 차이가 변동 폭보다 크다.")

print()
print("[E] 표 I 대 전부 연결 — H5")
a = dfE[dfE.model == "LeNet-5"].test_err.values
b = dfE[dfE.model == "C3 전부 연결"].test_err.values
w = max(a.max() - a.min(), b.max() - b.min())
print("  표 I %.2f%% (%s · 60,000) · 전부 연결 %.2f%% (%s · %s)"
      % (np.median(a), span(a), np.median(b), span(b), format(int(n_full), ",")))
print("  차이 %+.2f · 변동 폭 %.2f" % (np.median(b) - np.median(a), w))
print("  H5 판정:", "맞았다 — 차이가 변동 폭 안이라 순서를 매기지 않는다"
      if abs(np.median(b) - np.median(a)) <= w else
      "틀렸다 — 차이가 변동 폭보다 크다. 논문의 「대칭을 깬다」에 처음으로 숫자가 붙는다")

print()
print("시간: A %.0f 초 · D %.0f 초 · E %.0f 초 (seed 하나당 중앙값)"
      % (dfA.seconds.median(),
         dfD[dfD.model == "공유 끔"].seconds.median(),
         dfE[dfE.model == "C3 전부 연결"].seconds.median()))
print("=" * 88)